Case 1: Vivaldi Antenna Strong D2S

In [1]:
vivaldi_description = "We are going to model a Vivaldi antenna. It should consist of a substrate, a tapered slot (on the front of substrate), \
two rectangulars and a circle for feeding (on the back of substrate). The desired working frequency is from 3.0 - 13.5 GHz, so the substrate \
should be of 30x20 mm (W x L). Consider the left bottom corner under the substrate as origin of axis (0,0,0). \
The substrate should be of Rogers RO4003C of thickness ts (a fixed value 0.813), so the right top corner on the front is (30,20,ts). \
To model the tapered slot on the front, you need a full cover patch X-range [0,30], Y-range [0,20], Z-range [ts,ts+tp], \
tp should be a fixed value of 0.035. Then you need use the patch to substract a closed spline structure (extruded) and a circle. \
To form the slot, you need to model two symmetric splines and connect their top and bottom points to form a closed shape to extrude. \
The left-spline is defined by 20 points, whose y axis are 20, 19, ... 1, and the X axis should variables. (Here we will have 20 variables). \
The point's x axis should be ascending within the range (0, 15), exclusive. Meanwhile, the right half's X should be 30-X_i. \
Because the Vivaldi's tapered slot is complex, add step-by-step instruction to the description. \
The circle's center is located at（15, gap+r1), and its radius should be r1 (2 variables). \
On the back, the first rectangular starts from the right, and should be of X range [30-l1, 30], Y range [pf, pf+w1], Z range [-tp, 0], \
whose dimension is defined by l1 x w1 x tp (2 parameters, tp is defined before and is a fixed value). \
The second rectangular is of X range [30-l1-l2, 30-l1], Y range [pf + 0.5 * (w1 - w2), pf + 0.5 * (w1 + w2)], Z range [-tp, 0], \
whose dimension is of l2 x w2 x tp (2 parameters, tp is defined before and is a fixed value). From a geometry view, the second rectangular is connected \
to the end of the first retangular and align to the middle of it. \
Lastly, The circle on the back is a cylinder whose center is at (x,y) of (30-l1-l2, pf + 0.5 * w1) with radius of r2 (1 varaible) and Z range [-tp, 0]."

In [2]:
# Update import to match the actual filename
from leam.tools import StrongDescriptionToSolids

# Initialize StrongDescriptionToSolids tool with custom save directory (optional)
solids_generator = StrongDescriptionToSolids()

# Get CST solid specifications from the description and save as JSON
solids = solids_generator.get_solids(
    description=vivaldi_description,
    save_as="vivaldi.json"
)

print(solids)

{
  "solids": [
    {
      "Type": "3D",
      "name": "Substrate_RO4003C",
      "Role": "Dielectric",
      "material": "Rogers RO4003C",
      "dimensions": {
        "units": "mm",
        "origin_definition": "Global origin (0,0,0) is the left-bottom corner under the substrate.",
        "W": 30,
        "L": 20,
        "ts": 0.813,
        "Xrange": [
          0,
          30
        ],
        "Yrange": [
          0,
          20
        ],
        "Zrange": [
          0,
          "ts"
        ]
      },
      "operations": [],
      "notes": "Substrate brick. Top surface is at z=ts, bottom at z=0."
    },
    {
      "Type": "3D",
      "name": "FrontCopper",
      "Role": "Radiating",
      "material": "Copper",
      "dimensions": {
        "units": "mm",
        "tp": 0.035,
        "Xrange": [
          0,
          30
        ],
        "Yrange": [
          0,
          20
        ],
        "Zrange": [
          "ts",
          "ts+tp"
        ]
      },
      "ope

In [3]:
from leam.tools import ParameterGenerator

# Initialize ParameterGenerator with custom save directory
parameter_generator = ParameterGenerator()

# Generate parameters
parameters = parameter_generator.generate_parameters(
    description=vivaldi_description,
    output_file="vivaldi_parameters.bas",
    prompt_file="output/vivaldi.json"  # D2S output as prompt file
)


In [4]:
from leam.tools import DimensionGenerator
dimension = DimensionGenerator()

dimensions = dimension.generate_dimensions(
    description=vivaldi_description,
    additional_prompt_files=[
        "output/vivaldi.json",
        "output/vivaldi_parameters.bas"
    ],
    save_as = "vivaldi_dimensions.json"
    )

print(dimensions)


{
  "solids": [
    {
      "Type": "3D",
      "name": "Substrate_RO4003C",
      "reference": "Brick",
      "coordinates": {
        "x": 0,
        "y": 0,
        "z": 0
      },
      "dimensions": {
        "units": "mm",
        "material": "Rogers RO4003C",
        "W": 30,
        "L": 20,
        "ts": 0.813,
        "Xrange": [
          0,
          30
        ],
        "Yrange": [
          0,
          20
        ],
        "Zrange": [
          0,
          0.813
        ]
      },
      "notes": "Substrate brick. Xmin=0, Xmax=30, Ymin=0, Ymax=20, Zmin=0, Zmax=0.813. Global origin (0,0,0) is the left-bottom corner under the substrate."
    },
    {
      "Type": "3D",
      "name": "FrontCopper",
      "reference": "Brick",
      "coordinates": {
        "x": 0,
        "y": 0,
        "z": 0.813
      },
      "dimensions": {
        "units": "mm",
        "material": "Copper",
        "tp": 0.035,
        "Xrange": [
          0,
          30
        ],
        "Yran

In [5]:
from leam.tools import MaterialsProcessor
materials = MaterialsProcessor()

# Extract materials from the vivaldi.json file
material_names = materials.extract_materials("output/vivaldi_dimensions.json")

# Process material files from CST library
material_contents = materials.process_material_files(material_names)

# Generate VBA macro for material import
vba_code = materials.generate_vba_macro(
    material_contents,
    save_filename="vivaldi_materials.bas"
)

In [6]:
# Generate 3D model using Model3DGenerator
from leam.tools import Model3DGenerator

# Initialize Model3DGenerator with save directory
model_3d_generator = Model3DGenerator()

# Generate 3D model VBA code
model_code = model_3d_generator.generate_model(
    description=vivaldi_description,
    additional_prompt_files=[
        "output/vivaldi_parameters.bas",    # Parameters
        "output/vivaldi_dimensions.json",    # Dimensions
        "output/vivaldi_materials.bas"      # Materials
    ],
    save_as="vivaldi_model_3d.bas"
)

In [7]:
# Generate 2.5D model using Model2DGenerator
from leam.tools import Model2DGenerator

# Initialize Model2DGenerator with save directory
model_2d_generator = Model2DGenerator()

# Generate 2.5D model VBA code
model_code = model_2d_generator.generate_model(
    description=vivaldi_description,
    additional_prompt_files=[
        "output/vivaldi_parameters.bas",    # Parameters
        "output/vivaldi_dimensions.json",    # Dimensions
        "output/vivaldi_materials.bas",      # Materials
        "output/vivaldi_model_3d.bas"      # Materials
    ],
    save_as="vivaldi_model_2d.bas"
)

In [8]:
from leam.tools import BooleanOperationsGenerator

# Initialize BooleanOperationsGenerator with save directory
boolean = BooleanOperationsGenerator()

# Generate Boolean operations VBA code
boolean_code = boolean.generate_operations(
    description=vivaldi_description,
    additional_prompt_files=[
        "output/vivaldi_parameters.bas",    # Parameters
        "output/vivaldi_dimensions.json",    # Dimensions
        "output/vivaldi_model_2d.bas",     # 2.5D model
        "output/vivaldi_model_3d.bas"      # 3D model
    ],
    save_as="vivaldi_boolean.bas"
)

In [9]:
import os
from leam.tools import CstRunner

# Initialize CstRunner
runner = CstRunner(create_new_if_none=False)

# Create output directory in current folder if it doesn't exist
current_output = os.path.join(os.getcwd(), "output")
os.makedirs(current_output, exist_ok=True)

# Define tasks in execution order with correct path (history list)
vba_tasks = {
    "Parameters": os.path.join(current_output, "vivaldi_parameters.bas"),
    "Materials": os.path.join(current_output, "vivaldi_materials.bas"),
    "3D Model": os.path.join(current_output, "vivaldi_model_3d.bas"),
    "2.5D Model": os.path.join(current_output, "vivaldi_model_2d.bas"),
    "Boolean Operations": os.path.join(current_output, "vivaldi_boolean.bas"),
}

# Set history tasks
runner.set_history_tasks(vba_tasks)

# Create and save the CST project in the current output directory
project_path = os.path.join(current_output, "vivaldi_antenna.cst")
runner.create_project(project_path)

In [10]:
from leam.tools import ParameterUpdater

# Inputs
text_files = ["output/vivaldi_dimensions.json", "output/vivaldi_parameters.bas"]
update_para_description = (
    "I want to update parameters to form a Vivaldi shape "
    "And increase the retangulars length on the back and the radius of the cirle on the back."
)
save_filename = "vivaldi_update_parameters.bas"

# Generate update macro
updater = ParameterUpdater()
result = updater.generate_update(
    save_as=save_filename,
    description=update_para_description,
    additional_prompt_files=text_files,
)


In [11]:
import os
from leam.tools import CstRunner
from leam.utils import resolve_save_dir

# Execute parameter update via Schematic and save project
save_dir = resolve_save_dir()  # defaults to ./output
vba_path = os.path.join(save_dir, "vivaldi_update_parameters.bas")

project_path = os.path.join(save_dir, "vivaldi_antenna.cst")
runner = CstRunner(create_new_if_none=False, project_path=project_path)
runner.set_parameter_tasks({"Update Parameters": vba_path})
runner.apply_parameter_updates(
    save_path=os.path.join(save_dir, "vivaldi_update_parameters.cst"),
    include_results=False,
    allow_overwrite=True,
    close_project_after_save=True,
)

![Vivldi](assets/demo_images/Vivaldi_Front.jpg)



Case 2: Slotted Patch Antenna

In [12]:
slotted_patch_description = "I want to design a rectangular-slotted rectangular-patch antenna working at 2.45GHz."

In [13]:
# Update import to match the actual filename
from leam.tools import WeakDescriptionToSolids

# Initialize StrongDescriptionToSolids tool with custom save directory (optional)
solids_generator = WeakDescriptionToSolids()

# Get CST solid specifications from the description and save as JSON
solids = solids_generator.get_solids(
    description=slotted_patch_description,
    save_as="slotted_patch_antenna.json"
)

print(solids)

{
  "solids": [
    {
      "Type": "3D",
      "name": "GroundPlane",
      "Role": "Ground",
      "material": "Copper (annealed)",
      "dimensions": {
        "unit": "mm",
        "parameters": {
          "t_cu": 0.035,
          "Wsub": 60,
          "Lsub": 50
        },
        "brick": {
          "x1": "-Wsub/2",
          "x2": "Wsub/2",
          "y1": "-Lsub/2",
          "y2": "Lsub/2",
          "z1": 0,
          "z2": "t_cu"
        }
      },
      "operations": [],
      "notes": "Bottom copper ground plane. Substrate will start at z=t_cu to avoid overlap."
    },
    {
      "Type": "3D",
      "name": "Substrate",
      "Role": "Dielectric",
      "material": "FR-4 (lossy)",
      "dimensions": {
        "unit": "mm",
        "parameters": {
          "t_cu": 0.035,
          "h_sub": 1.6,
          "Wsub": 60,
          "Lsub": 50
        },
        "brick": {
          "x1": "-Wsub/2",
          "x2": "Wsub/2",
          "y1": "-Lsub/2",
          "y2": "Lsub/2

In [14]:
from leam.tools import ParameterGenerator

# Initialize ParameterGenerator with custom save directory
parameter_generator = ParameterGenerator()

# Generate parameters
parameters = parameter_generator.generate_parameters(
    description=slotted_patch_description,
    output_file="slotted_patch_parameters.bas",
    prompt_file="output/slotted_patch_antenna.json" # D2S output as prompt file
)


In [15]:
from leam.tools import DimensionGenerator
dimension = DimensionGenerator()

dimensions = dimension.generate_dimensions(
    description=slotted_patch_description,
    additional_prompt_files=[
        "output/slotted_patch_antenna.json",
        "output/slotted_patch_parameters.bas"
    ],
    save_as = "slotted_patch_dimensions.json"
    )

print(dimensions)

{
  "solids": [
    {
      "Type": "3D",
      "name": "GroundPlane",
      "reference": "Brick",
      "coordinates": {
        "x": 0,
        "y": 0,
        "z": 0
      },
      "dimensions": {
        "unit": "mm",
        "parameters": {
          "t_cu": 0.035,
          "Wsub": 60,
          "Lsub": 50
        },
        "brick": {
          "Xmin": "-Wsub/2",
          "Xmax": "Wsub/2",
          "Ymin": "-Lsub/2",
          "Ymax": "Lsub/2",
          "Zmin": "0",
          "Zmax": "t_cu"
        }
      },
      "notes": "Material: Copper (annealed). Brick extents: Xmin=-Wsub/2, Xmax=+Wsub/2, Ymin=-Lsub/2, Ymax=+Lsub/2, Zmin=0, Zmax=t_cu."
    },
    {
      "Type": "3D",
      "name": "Substrate",
      "reference": "Brick",
      "coordinates": {
        "x": 0,
        "y": 0,
        "z": 0
      },
      "dimensions": {
        "unit": "mm",
        "parameters": {
          "t_cu": 0.035,
          "h_sub": 1.6,
          "Wsub": 60,
          "Lsub": 50,
          "

In [16]:
from leam.tools import MaterialsProcessor
materials = MaterialsProcessor()

# Extract materials from the vivaldi.json file
material_names = materials.extract_materials("output/slotted_patch_dimensions.json")

# Process material files from CST library
material_contents = materials.process_material_files(material_names)

# Generate VBA macro for material import
vba_code = materials.generate_vba_macro(
    material_contents,
    save_filename="slotted_patch_materials.bas"
)


In [17]:
# Generate 3D model using Model3DGenerator
from leam.tools import Model3DGenerator

# Initialize Model3DGenerator with save directory
model_3d_generator = Model3DGenerator()

# Generate 3D model VBA code
model_code = model_3d_generator.generate_model(
    description=slotted_patch_description,
    additional_prompt_files=[
        "output/slotted_patch_parameters.bas",    # Parameters
        "output/slotted_patch_dimensions.json",    # Dimensions
        "output/slotted_patch_materials.bas"      # Materials
    ],
    save_as="slotted_patch_model.bas"
)


In [18]:
from leam.tools import BooleanOperationsGenerator

# Initialize BooleanOperationsGenerator with save directory
boolean = BooleanOperationsGenerator()

# Generate Boolean operations VBA code
boolean_code = boolean.generate_operations(
    description=slotted_patch_description,
    additional_prompt_files=[
        "output/slotted_patch_parameters.bas",    # Parameters
        "output/slotted_patch_dimensions.json",    # Dimensions
        "output/slotted_patch_model.bas",     # 3D model
    ],
    save_as="slotted_patch_boolean.bas"
)

In [19]:
import os
from leam.tools import CstRunner

# Initialize CstRunner
runner = CstRunner(create_new_if_none=False)

# Create output directory in current folder if it doesn't exist
current_output = os.path.join(os.getcwd(), "output")
os.makedirs(current_output, exist_ok=True)

# Define tasks in execution order with correct path (history list)
vba_tasks = {
    "Parameters": os.path.join(current_output, "slotted_patch_parameters.bas"),
    "Materials": os.path.join(current_output, "slotted_patch_materials.bas"),
    "3D Model": os.path.join(current_output, "slotted_patch_model.bas"),
    "Boolean Operations": os.path.join(current_output, "slotted_patch_boolean.bas"),
}

# Set history tasks
runner.set_history_tasks(vba_tasks)

# Create and save the CST project in the current output directory
project_path = os.path.join(current_output, "slotted_patch_antenna.cst")
runner.create_project(project_path)

![Slotted_Patch](assets/demo_images/Slotted_Patch.jpg)

![Slotted_Patch_S11](assets/demo_images/Slotted_Patch_S11.jpg)

Case 3: Slotted Monopole Antenna

In [20]:
monopole_description = "The layout of the slotted monopole antenna is shown in Fig. 4. The antenna is implemented on an FR-4 substrate with a thickness of 0.8 mm, a relative permittivity of 4.4, and a loss tangent of 0.02. It consists of a driven circular patch radiator and two uniform rectangular metal planes separated by the microstrip line. Two slots are fused at the center of the driven circular patch radiator to form a quasi-cross slot, and the geometry of the slot helps control the surface current distribution. Meanwhile, the rectangular planes act as a coplanar partial ground."  # Additional description

In [ ]:
# Update import to match the actual filename
from leam.tools import StrongDescriptionToSolids

# Initialize StrongDescriptionToSolids tool with custom save directory (optional)
solids_generator = StrongDescriptionToSolids()

# Get CST solid specifications from the description and save as JSON
solids = solids_generator.get_solids(
    image_paths=["assets/Monopole.gif"],
    description=monopole_description,
    save_as="monopole.json"
)

print(solids)

In [ ]:
from leam.tools import ParameterGenerator

# Initialize ParameterGenerator with custom save directory
parameter_generator = ParameterGenerator()

monopole_param_description = "There are 12 variables. \
There is an error in the graph, \
SL should equal to *** ML + DPR + 0.2 ***."

# Generate parameters
parameters = parameter_generator.generate_parameters(
    image_paths=["assets/Monopole_para.gif"],
    description=monopole_param_description,
    output_file="monopole_parameters.bas",
    prompt_file="output/monopole.json"  # D2S output as prompt file
)


In [ ]:
from leam.tools import DimensionGenerator
dimension = DimensionGenerator()

monopole_dimension_description = "SLH is the length of the horizontal slot on the x-axis. \
SLV is the length of the vertical slot on the y-axis. \
SLT is the width of both the rectangular slots. \
Two slots should be centered at the center of circle. \
IMPORTANT: RPL definition is not straightforward, the length of RP should be ML-RPL. \
The circle's center is at (SW/2, ML). The patch and the ground planes are all on the substrate."

dimensions = dimension.generate_dimensions(
    description=monopole_dimension_description,
    image_paths=["assets/Monopole.gif"],
    additional_prompt_files=[
        "output/monopole.json",
        "output/monopole_parameters.bas"
    ],
    save_as = "monopole_dimensions.json"
    )

print(dimensions)

{
  "solids": [
    {
      "Type": "3D",
      "name": "Substrate_FR4",
      "reference": "Brick",
      "coordinates": {
        "x": 0,
        "y": 0,
        "z": 0
      },
      "dimensions": {
        "units": "mm",
        "params": {
          "SW": 14.4,
          "SL": 33.55,
          "h_sub": 0.8
        },
        "ranges": {
          "x": [
            0,
            14.4
          ],
          "y": [
            0,
            33.55
          ],
          "z": [
            0,
            0.8
          ]
        }
      },
      "notes": "Material: FR-4. Brick ranges: Xmin=0, Xmax=SW(14.4), Ymin=0, Ymax=SL(33.55), Zmin=0, Zmax=h_sub(0.8). Global origin at substrate bottom-left corner on bottom face (0,0,0); +z upwards."
    },
    {
      "Type": "3D",
      "name": "CPW_Ground_Left",
      "reference": "Brick",
      "coordinates": {
        "x": 0,
        "y": 0,
        "z": 0.8
      },
      "dimensions": {
        "units": "mm",
        "params": {
          "

In [ ]:
from leam.tools import MaterialsProcessor
materials = MaterialsProcessor()

# Extract materials from the vivaldi.json file
material_names = materials.extract_materials("output/monopole_dimensions.json")

# Process material files from CST library
material_contents = materials.process_material_files(material_names)

# Generate VBA macro for material import
vba_code = materials.generate_vba_macro(
    material_contents,
    save_filename="monopole_materials.bas"
)


In [ ]:
# Generate 3D model using Model3DGenerator
from leam.tools import Model3DGenerator

# Initialize Model3DGenerator with save directory
model_3d_generator = Model3DGenerator()

# Generate 3D model VBA code
model_code = model_3d_generator.generate_model(
    additional_prompt_files=[
        "output/monopole_parameters.bas",    # Parameters
        "output/monopole_dimensions.json",    # Dimensions
        "output/monopole_materials.bas"      # Materials
    ],
    save_as="monopole_model.bas"
)

In [ ]:
from leam.tools import BooleanOperationsGenerator
Monopole_boolean = "We need add the feed to the patch then substrate slots from the patch."

# Initialize BooleanOperationsGenerator with save directory
boolean = BooleanOperationsGenerator()

# Generate Boolean operations VBA code
boolean_code = boolean.generate_operations(
    description=Monopole_boolean,
    additional_prompt_files=[
        "output/monopole_parameters.bas",    # Parameters
        "output/monopole_dimensions.json",    # Dimensions
        "output/monopole_model.bas",     # 3D model
    ],
    save_as="monopole_boolean.bas"
)

In [ ]:
import os
from leam.tools import CstRunner

# Initialize CstRunner
runner = CstRunner(create_new_if_none=False)

# Create output directory in current folder if it doesn't exist
current_output = os.path.join(os.getcwd(), "output")
os.makedirs(current_output, exist_ok=True)

# Define tasks in execution order with correct path (history list)
vba_tasks = {
    "Parameters": os.path.join(current_output, "monopole_parameters.bas"),
    "Materials": os.path.join(current_output, "monopole_materials.bas"),
    "3D Model": os.path.join(current_output, "monopole_model.bas"),
    "Boolean Operations": os.path.join(current_output, "monopole_boolean.bas"),
}

# Set history tasks
runner.set_history_tasks(vba_tasks)

# Create and save the CST project in the current output directory
project_path = os.path.join(current_output, "monopole_antenna.cst")
runner.create_project(project_path)


In [ ]:
from leam.tools import ParameterUpdater

# Define your inputs
text_files = ["output/monopole_dimensions.json", "output/monopole_parameters.bas"]

update_para_description = (
    "I want to demonstrate with the following parameters."
    "$DP_{R}$ = 6.58, $S_{W}$ = 13.43, $SL_{T}$ = 1, $SL_{V}$ = 7.9, $SL_{H}$ = 7.9, "
    "$M_{L}$ = 25.08, $RP_{L}$ = 6.67, $M_{W}$ = 1.2, $M_{G}$ = 0.3, $S_{L}$ = 31.86, $RP_{W}$ = 5.815. (unit: mm.)}"
)

save_filename = "monopole_update_parameters.bas"

# Create UpdateParameter instance
updater = ParameterUpdater()

# Call generate_update
result = updater.generate_update(
    save_as=save_filename,
    description=update_para_description,
    additional_prompt_files=text_files,
)

In [ ]:
import os
from leam.tools import CstRunner

project_path = os.path.join(os.getcwd(), "output", "monopole_antenna.cst")
vba_path = os.path.join(os.getcwd(), "output", "monopole_update_parameters.bas")

runner = CstRunner(create_new_if_none=False, project_path=project_path)
runner.set_parameter_tasks({"Update Parameters": vba_path})
runner.apply_parameter_updates(
    save_path=None,               # no save, just execute
    close_project_after_save=True,
)


![monopole](assets/demo_images/monopole.jpg)